In [ ]:
import json
import re
import csv
import pandas as pd
import io
import os
from pyvi import ViTokenizer

In [ ]:
# --- CÁC HÀM XỬ LÝ ---

def load_and_prepare_dictionary(json_path):
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        normalization_dict = {key: value[0] for key, value in json_data.items()}
        sorted_keys = sorted(normalization_dict.keys(), key=len, reverse=True)
        return {key: normalization_dict[key] for key in sorted_keys}
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file từ điển tại '{json_path}'.")
        return {}
    except Exception as e:
        print(f"Đã xảy ra lỗi khi đọc file JSON: {e}")
        return {}


def normalize_text_with_json_dict(text, normalization_dict):
    text = text.lower()
    for wrong_word, correct_word in normalization_dict.items():
        wrong_word_escaped = re.escape(wrong_word)
        text = re.sub(r'\b' + wrong_word_escaped + r'\b', correct_word, text)

    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[\U00010000-\U0010ffff]', '', text)

    # Xóa các ký tự đặc biệt
    text = re.sub(r'[^\w\s,.?!À-ỹ]', '', text)

    # Xóa các dấu câu bị lặp lại
    text = re.sub(r'([,.?!])\1+', r'\1', text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def process_dataset_to_csv(input_file, output_file, normalization_dict):
    if not normalization_dict:
        print("Từ điển rỗng, không thể chuẩn hóa.")
        return

    try:
        with open(input_file, 'r', encoding='utf-8') as f_in, \
             open(output_file, 'w', encoding='utf-8', newline='') as f_out:
            writer = csv.writer(f_out)
            writer.writerow(['normalized_text'])
            for line in f_in:
                normalized_line = normalize_text_with_json_dict(line, normalization_dict)
                if normalized_line:
                    writer.writerow([normalized_line])
        print(f"Đã chuẩn hóa thành công và lưu vào file CSV: {output_file}")
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file '{input_file}'.")
    except Exception as e:
        print(f"Đã xảy ra lỗi: {e}")


In [ ]:
# --- THỰC THI ---

# THAY TÊN FILE
file_name_dict = "vi-nsw-dict.json"
input_dataset_file = "Final_data.csv"
output_dataset_file = "normalized_dataset.csv"

my_dictionary = load_and_prepare_dictionary(file_name_dict)

if my_dictionary:
    process_dataset_to_csv(input_dataset_file, output_dataset_file, my_dictionary)

    print(f"\n--- XEM TRƯỚC NỘI DUNG FILE '{output_dataset_file}' ---")
    try:
        df = pd.read_csv(output_dataset_file)
        display(df)
    except pd.errors.EmptyDataError:
        print("File CSV rỗng, không có dữ liệu để hiển thị.")

✅ Đã chuẩn hóa thành công và lưu vào file CSV: normalized_dataset.csv

--- XEM TRƯỚC NỘI DUNG FILE 'normalized_dataset.csv' ---


,normalized_text
0,"comment,label"
1,"shop phục vụ rất kém,neg"
2,"tôi về vất đi rồi, chỉ được quảng cáo hay thôi..."
3,chất lượng sản phẩm rất kém đóng gói sản phẩm ...
4,"bề ngang áo chật,neg"
...,...
15166,"mỗi mềm hơn rất nhju,pos"
15167,"cực kì đáng tiền,pos"
15168,"rẻ, đẹp trai,pos"
15169,"chất lượng, màu sắc khá ổn, giống ảnh. khá hài..."


In [ ]:
df = pd.read_csv("normalized_dataset.csv")
df.head()

,normalized_text
0,"comment,label"
1,"shop phục vụ rất kém,neg"
2,"tôi về vất đi rồi, chỉ được quảng cáo hay thôi..."
3,chất lượng sản phẩm rất kém đóng gói sản phẩm ...
4,"bề ngang áo chật,neg"


In [ ]:
source_column_name = 'normalized_text'

# Bước 1: Bỏ dòng đầu tiên và reset index
df_cleaned = df.iloc[1:].reset_index(drop=True)

# Bước 2: Tách cột 'normalized_text' thành 2 cột mới 'comment' và 'label'
df_cleaned[['comment', 'label']] = df_cleaned[source_column_name].str.rsplit(',', n=1, expand=True)

# Bước 3: Xóa cột 'normalized_text' đi
df_cleaned = df_cleaned.drop(columns=[source_column_name])

# Bước 4:Xóa khoảng trắng thừa ở cột nhãn
df_cleaned['label'] = df_cleaned['label'].str.strip()


print("Dữ liệu đã được tách thành công:")
print(df_cleaned.head())

df = df_cleaned

Dữ liệu đã được tách thành công:
                                             comment label
0                               shop phục vụ rất kém   neg
1  tôi về vất đi rồi, chỉ được quảng cáo hay thôi...   neg
2  chất lượng sản phẩm rất kém đóng gói sản phẩm ...   neg
3                                   bề ngang áo chật   neg
4                             và hẳn dây thì tai mèo   neg


In [ ]:
df.shape

(15170, 2)

In [ ]:
allowed_labels = ['neu', 'pos', 'neg']

df_filtered = df[df['label'].isin(allowed_labels)]

print(f"Số hàng ban đầu: {len(df)}")
print(f"Số hàng sau khi lọc: {len(df_filtered)}")
print("\nDataFrame sau khi đã xóa các hàng không hợp lệ:")
print(df_filtered.head())

Số hàng ban đầu: 15170
Số hàng sau khi lọc: 14999

DataFrame sau khi đã xóa các hàng không hợp lệ:
                                             comment label
0                               shop phục vụ rất kém   neg
1  tôi về vất đi rồi, chỉ được quảng cáo hay thôi...   neg
2  chất lượng sản phẩm rất kém đóng gói sản phẩm ...   neg
3                                   bề ngang áo chật   neg
4                             và hẳn dây thì tai mèo   neg


In [ ]:
output_filename = 'Data_Normalized.csv'
df_filtered.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"Đã xuất file thành công với tên: {output_filename}")

Đã xuất file thành công với tên: Data_Normalized.csv


# segmentation

In [ ]:
file_path = 'Data_Normalized.csv'
try:
    df = pd.read_csv(file_path)
    print("Đọc file thành công!")
    print("5 dòng đầu của dữ liệu:")
    print(df.head())
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file '{file_path}'.")
    print("Vui lòng kiểm tra lại tên file hoặc tải file lên Colab trước.")


text_column = 'comment'

if text_column in df.columns:
    print(f"\nBắt đầu tách từ cho cột '{text_column}'...")
    df['tokenized_text'] = df[text_column].astype(str).apply(ViTokenizer.tokenize)
    print("Tách từ hoàn tất!")
else:
    print(f"Lỗi: Không tìm thấy cột '{text_column}' trong DataFrame.")

Đọc file thành công!
5 dòng đầu của dữ liệu:
                                             comment label
0                               shop phục vụ rất kém   neg
1  tôi về vất đi rồi, chỉ được quảng cáo hay thôi...   neg
2  chất lượng sản phẩm rất kém đóng gói sản phẩm ...   neg
3                                   bề ngang áo chật   neg
4                             và hẳn dây thì tai mèo   neg

Bắt đầu tách từ cho cột 'comment'...
Tách từ hoàn tất!


In [ ]:
df.head(20)

,comment,label,tokenized_text
0,shop phục vụ rất kém,neg,shop phục_vụ rất kém
1,"tôi về vất đi rồi, chỉ được quảng cáo hay thôi...",neg,"tôi về vất đi rồi , chỉ được quảng_cáo hay thô..."
2,chất lượng sản phẩm rất kém đóng gói sản phẩm ...,neg,chất_lượng sản_phẩm rất kém đóng_gói sản_phẩm ...
3,bề ngang áo chật,neg,bề ngang áo chật
4,và hẳn dây thì tai mèo,neg,và hẳn dây thì tai mèo
5,chuẩn bị hàng lâu,neg,chuẩn_bị hàng lâu
6,áo hơi mỏng không giống chất liệu len,neg,áo hơi mỏng không giống chất_liệu len
7,giao hàng chậm,neg,giao hàng chậm
8,mặc em những mặc 34 lấy size sao hơi chật,neg,mặc em những mặc 34 lấy size sao hơi chật
9,"bung chỉ, vãi tạm",neg,"bung chỉ , vãi tạm"


In [ ]:
output_filename = 'Data_Tokenization.csv'
df.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"Đã xuất file thành công với tên: {output_filename}")

Đã xuất file thành công với tên: Data_Tokenization.csv


# Loại bỏ STOPWORDS

In [ ]:
dataset_filename = 'Data_Tokenization.csv'
stopwords_filename = 'stopwords-vi.txt'

def load_stopwords(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            stopwords = {line.strip() for line in f if line.strip()}
        print(f"Đã tải và xử lý {len(stopwords)} stopwords từ '{filepath}'.")
        return stopwords
    except FileNotFoundError:
        print(f"LỖI: Không tìm thấy tệp stopwords '{filepath}'.")
        return None

def remove_stopwords(text, stopwords_set):
    """Loại bỏ stopwords khỏi một chuỗi văn bản."""
    if not isinstance(text, str):
        return ""

    words = text.lower().split()
    filtered_words = [word for word in words if word not in stopwords_set]
    return ' '.join(filtered_words)

In [ ]:
# Tải stopwords
vietnamese_stopwords = load_stopwords(stopwords_filename)

if vietnamese_stopwords is not None:
    try:
        # Đọc dataset từ tệp đã có trong Colab
        df = pd.read_csv(dataset_filename)
        print(f"Đã tải dataset '{dataset_filename}' với {len(df)} dòng.")

        TEXT_COLUMN = 'comment'

        # Kiểm tra cột 'comment' có tồn tại không
        if TEXT_COLUMN in df.columns:
            print(f"\nBắt đầu xử lý cột '{TEXT_COLUMN}'...")

            # Áp dụng hàm và tạo cột mới
            df['comment'] = df[TEXT_COLUMN].apply(lambda x: remove_stopwords(x, vietnamese_stopwords))

            print("Xử lý hoàn tất!")


            # Lưu và tải tệp kết quả
            output_filename = 'Data_Completed.csv'
            df.to_csv(output_filename, index=False, encoding='utf-8-sig')
            print(f"\nĐã lưu kết quả vào '{output_filename}'. Bắt đầu tải xuống...")

            from google.colab import files
            files.download(output_filename)

        else:
            print(f"LỖI: Không tìm thấy cột '{TEXT_COLUMN}' trong tệp của bạn.")
            print(f"Các cột hiện có là: {list(df.columns)}")

    except FileNotFoundError:
        print(f"LỖI: Không tìm thấy tệp dataset '{dataset_filename}'.")

Đã tải và xử lý 645 stopwords từ 'stopwords-vi.txt'.
Đã tải dataset 'Data_Tokenization.csv' với 14999 dòng.

Bắt đầu xử lý cột 'comment'...
Xử lý hoàn tất!

Đã lưu kết quả vào 'Data_Completed.csv'. Bắt đầu tải xuống...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>